# BIOM 200: Our nasal microbiomes

Instructors: Rob Knight (robknight@ucsd.edu)

On Thursday, October 15, we swabbed the left and right nostril of all willing participants, including both students and instructors. The swabs were extracted, amplified for the 16S rRNA V4 region, sequenced the same day, and processed in Qiita (study [16596](https://qiita.ucsd.edu/study/description/16596)). This notebook asks four questions of those data:

1. **Who lives in our noses?** Which bacteria dominate, and is it the same in everyone?
2. **Left vs. right:** do the two nostrils of the same person host different communities?
3. **Self vs. others:** is your left nostril more like your right nostril than like anyone else's?
4. **Students vs. instructors:** do the two groups differ?

Before running anything, think about what you expect for each question and why.

## The nasal microbiome

The anterior nares are dominated by a small number of genera, typically *Corynebacterium*, *Cutibacterium*, *Staphylococcus*, *Dolosigranulum*, and *Moraxella*, and *Staphylococcus aureus* is carried by a substantial fraction of healthy adults. Body site explains more of the variation in the human microbiome than anything else, but within a site, individuals differ consistently from one another. Nasal swabs are also low in biomass, so contamination from reagents is a real risk.

**Read more:** [Bomar et al. 2018, *Curr Opin Microbiol*](https://doi.org/10.1016/j.mib.2017.10.023) · [Human Microbiome Project Consortium 2012, *Nature*](https://doi.org/10.1038/nature11234) · [Costello et al. 2009, *Science*](https://doi.org/10.1126/science.1177486)

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
from scripts.biom200 import *   # autoreload picks up edits to the module without a restart

## Load the class data

The class data (Qiita study 16596) are included in `data/class/`, with sample labels corrected against the lab's plate map. Participants are identified only by codes (P01–P18).

In [2]:
TABLE = "data/class/16596_feature-table.biom"
METADATA = "data/class/16596_sample_information.tsv"
TAXONOMY = "data/class/16596_taxonomy.tsv"
TREE = "data/class/16596_tree.nwk"

INSTRUCTORS = ["P15", "P04", "P10", "P18"]   # participant codes of the instructors; everyone else is a student

nasal = load_dataset("nasal", TABLE, METADATA, TAXONOMY, TREE, group="side", subject="host_subject_id",
                     control_col="sample_type", control_values=["control blank"])
add_metadata_column(nasal, "role", lambda m: np.where(m["host_subject_id"].isin(INSTRUCTORS), "instructor", "student"))

nasal: 91 samples and 1812 features; 91 samples have metadata
  5 samples in the metadata have no reads in the table
  56 controls (control blank): median 460 reads vs 49,316 in samples
  35 biological samples retained; fields available for grouping: ['side']
role
student       27
instructor     8


## Question 1: who lives in our noses?

Start with the bar plot: which genera dominate, and do they match the expected nasal genera listed above? Switch **Level** to *Species* to see how far 16S can resolve them (some, such as *Staphylococcus aureus* and *S. epidermidis*, are hard to separate with the V4 region). Group by `side` or `role`, and tick **Group means** to compare averages.

**Read more:** [Bomar et al. 2018, *Curr Opin Microbiol*](https://doi.org/10.1016/j.mib.2017.10.023)

In [4]:
add_composition(nasal)
explore_composition(nasal)

interactive(children=(Dropdown(description='Level', index=4, options=('Phylum', 'Class', 'Order', 'Family', 'G…

The table lists the most abundant genera: their mean share of reads, how many samples and how many people carry them, and in how many people each one is the most abundant genus. Is the nasal microbiome the same "core" in everyone, or does each person have their own dominant member?

In [6]:
top_taxa(nasal, "Genus")

,mean relative abundance,samples carrying it,host_subject_ids carrying it,dominant in (of 18 host_subject_ids)
Genus,,,,
Corynebacterium,43.1%,100.0%,100.0%,15
Staphylococcus,12.1%,94.3%,94.4%,0
Lawsonella,11.8%,97.1%,100.0%,1
Dolosigranulum,4.7%,97.1%,100.0%,0
Streptococcus,4.3%,97.1%,100.0%,1
Neisseriaceae (f.),3.9%,94.3%,94.4%,0
Anaerococcus,3.2%,94.3%,94.4%,0
Peptoniphilus_A,2.5%,94.3%,94.4%,0
Escherichia,1.8%,68.6%,83.3%,1


The same at species level. Treat species names from 16S with care: a 150-nucleotide fragment often cannot tell close relatives apart, and the reference assigns a species only when it can.

In [7]:
top_taxa(nasal, "Species")

,mean relative abundance,samples carrying it,host_subject_ids carrying it,dominant in (of 18 host_subject_ids)
Species,,,,
Corynebacterium accolens,24.7%,100.0%,100.0%,6
Corynebacterium kefirresidentii,13.6%,77.1%,77.8%,3
Staphylococcus capitis,11.9%,94.3%,94.4%,2
Lawsonella clevelandensis,11.8%,97.1%,100.0%,3
Dolosigranulum pigrum,4.7%,97.1%,100.0%,0
Corynebacterium pseudodiphtheriticum,4.0%,97.1%,100.0%,1
Neisseriaceae (f.),3.9%,94.3%,94.4%,1
Peptoniphilus_A senegalensis,2.4%,94.3%,94.4%,0
Streptococcus thermophilus,2.3%,94.3%,94.4%,1


**What our noses show.** *Corynebacterium* is in every one of us and makes up at least 5% of everyone's reads (43% on average); *C. accolens* is the most common species. *Staphylococcus* and *Lawsonella* follow at about 12% each, and *Lawsonella* reaches over half of one person's reads. *Dolosigranulum* is in everyone but abundant in only a few. In contrast, *Cutibacterium* and *Moraxella*, often listed among the typical nasal genera, are detectable in most of us but never above 0.5% of anyone's reads.

Why might our class differ from the published picture? Consider where exactly the swab sampled (the skin-like nostril entrance or deeper in the nasal cavity), how readily different bacteria release DNA during extraction, how well the V4 primers match each group, and who was sampled (adults, at one time point, in one place).

## Controls: what do the blanks tell us?

Now that you know who lives in our noses, check which of those reads could have come from somewhere else. Each extraction plate carried blanks: wells with no sample, processed exactly like the swabs. Reads in a blank come from reagents, from the lab, or from neighbouring wells. Compare the blanks with the swabs: how many reads do they have, and which taxa? A blank with as many reads as a real sample, or containing the dominant taxa of a neighbouring sample, points to well-to-well contamination or a swapped tube rather than reagent background.

**Read more:** [Salter et al. 2014, *BMC Biol*](https://doi.org/10.1186/s12915-014-0087-z) · [Minich et al. 2019, *mSystems*](https://doi.org/10.1128/mSystems.00186-19)

In [8]:
explore_controls(nasal)

interactive(children=(Dropdown(description='Level', index=4, options=('Phylum', 'Class', 'Order', 'Family', 'G…

In [9]:
control_report(nasal).head(10)

,control,reads,top taxa
0,16596.BLANK.RobOct2026Class.1.F8,8260,"Sphingomicrobium ginkgonis (17%), Sphingomonas..."
1,16596.BLANK.RobOct2026Class.1.G12,2450,"Acidovorax_F soli (24%), Methylophilaceae (f.)..."
2,16596.BLANK.RobOct2026Class.1.H12,1987,"Acidovorax_F soli (28%), Methylophilaceae (f.)..."
3,16596.BLANK.RobOct2026Class.1.A6,1593,"Ezakiella coagulans (7%), Staphylococcus capit..."
4,16596.BLANK.RobOct2026Class.1.E5,958,"Caldora sp010672925 (91%), Corynebacterium acc..."
5,16596.BLANK.RobOct2026Class.1.C7,956,"Marinagarivorans (24%), Curvibacter lanceolatu..."
6,16596.BLANK.RobOct2026Class.1.D5,947,"Methylobacterium gregans (26%), Burkholderia u..."
7,16596.BLANK.RobOct2026Class.1.C3,939,"Ralstonia pickettii (19%), Methylobacterium gr..."
8,16596.BLANK.RobOct2026Class.1.D2,896,"Acidovorax_F soli (24%), Burkholderia ubonensi..."
9,16596.BLANK.RobOct2026Class.1.H7,894,"Acidovorax_F soli (16%), Curvibacter lanceolat..."


## Rarefaction

Read depth varies over several orders of magnitude between our samples. Choose a depth where most curves have flattened, and note which samples fall below it. Losing one nostril of a person also removes that person from the paired comparisons below.

**Read more:** [Weiss et al. 2017, *Microbiome*](https://doi.org/10.1186/s40168-017-0237-y)

In [10]:
add_curves(nasal)
explore_rarefaction(nasal)

interactive(children=(Dropdown(description='Colour by', options=('side', 'role'), style=DescriptionStyle(descr…

In [11]:
add_diversity(nasal)

Rarefying to 2,800 reads; 4 of 35 samples dropped


/opt/homebrew/Caskroom/miniforge/base/envs/rachis-qiime2-2026.7/lib/python3.12/site-packages/skbio/util/_warning.py:42: DeprecationWarning: `observed_otus` was renamed to `sobs` in 0.6.0. The old name is kept as an alias but is deprecated.
  warn(message, warning)


Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command:

faithpd -i /var/folders/9k/dg_fpv9j08dd5_7mzskwwbfh0000gn/T/qiime2/lucas/processes/14412-1792425050.214155@lucas/tmp/rachis-OutPath-d6rmidns -t /var/folders/9k/dg_fpv9j08dd5_7mzskwwbfh0000gn/T/qiime2/lucas/data/f189e905-b38a-4c32-9b3a-fb9e8b094939/data/tree.nwk -o /var/folders/9k/dg_fpv9j08dd5_7mzskwwbfh0000gn/T/qiime2/lucas/processes/14412-1792425050.214155@lucas/tmp/rachis-OutPath-to5jtipm

Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command:

ssu -i /var/folders/9k/dg_fpv9j08dd5_7mzskwwbfh0000gn/T/qiime2/lucas/data/a38886c8-53c7-4efd-ae41-e7bb86444adc/data/feature-table.biom -t 

observed_features        shannon       faith_pd         
                   mean    std    mean   std     mean      std
side                                                          
left              68.41  22.87    3.14  0.87  4119.64  1010.31
right             77.07  31.16    3.06  0.89  4407.75  1344.17

## Question 2: left vs. right

Each person contributes one left and one right sample, so the two sides are **paired**. A paired test compares each person's left with their own right, which removes the large differences between people from the comparison. Each line below joins one person's two nostrils.

In [12]:
explore_paired_alpha(nasal, group="side", pair_by="host_subject_id")

interactive(children=(Dropdown(description='Metric', index=1, options=('observed_features', 'shannon', 'faith_…

The same question for the whole community: in the PCoA, grey lines join the two nostrils of each person. Short lines mean the two sides are similar. Tick **Label host_subject_id** to name each point. Note that PERMANOVA by `side` treats all samples as independent; with paired samples, the comparison in Question 3 is more informative.

In [13]:
explore_pcoa(nasal, connect="host_subject_id")

interactive(children=(Dropdown(description='Metric', options=('bray_curtis', 'jaccard', 'unweighted_unifrac', …

## Question 3: are you more like yourself than like anyone else?

Compare the distance between your own two nostrils with the distances between nostrils of different people. If individuals have consistent nasal communities, same-person distances should be smaller.

**Read more:** [Costello et al. 2009, *Science*](https://doi.org/10.1126/science.1177486) · [Lozupone et al. 2011, *ISME J*](https://doi.org/10.1038/ismej.2010.133)

In [14]:
explore_within_between(nasal, by="host_subject_id")

interactive(children=(Dropdown(description='Metric', options=('bray_curtis', 'jaccard', 'unweighted_unifrac', …

In [15]:
explore_distance_matrix(nasal)

interactive(children=(Dropdown(description='Metric', options=('bray_curtis', 'jaccard', 'unweighted_unifrac', …

PERMANOVA for side, participant, and role. Participant usually explains far more than side.

**Read more:** [Anderson 2001, *Austral Ecol*](https://doi.org/10.1111/j.1442-9993.2001.01070.pp.x)

In [16]:
permanova_table(nasal, ["side", "host_subject_id", "role"])

,metric,grouping,pseudo-F,p
0,bray_curtis,side,0.27,0.992
1,bray_curtis,host_subject_id,7.40,0.001
2,bray_curtis,role,2.86,0.022
3,jaccard,side,0.76,0.995
4,jaccard,host_subject_id,1.54,0.001
5,jaccard,role,1.18,0.069
6,unweighted_unifrac,side,0.64,0.983
7,unweighted_unifrac,host_subject_id,1.59,0.001
8,unweighted_unifrac,role,1.09,0.270
9,weighted_unifrac,side,0.10,0.998


## Question 4: students vs. instructors

Use **Group by → role** in the plots above, or the cells below. Each person contributes two samples, so the samples are not independent: with *n* people, there are only *n* independent observations, not 2*n*. Be cautious with any p-value here, and ask what else differs between the two groups besides being an instructor.

In [18]:
explore_alpha(nasal)

interactive(children=(Dropdown(description='Metric', index=1, options=('observed_features', 'shannon', 'faith_…

## Differential abundance

First, left vs. right, then students vs. instructors. Both tests treat every sample as independent. Adjusting for participant would compare each person with themselves, but with only a few samples per person that model has too little data left to estimate variability and reports many false positives (in checks with shuffled labels, most runs found "significant" genera where there were none); the paired plots above handle the pairing instead. With this few people, treat every result as a hypothesis.

**Read more:** [Lin & Peddada 2020, *Nat Commun*](https://doi.org/10.1038/s41467-020-17041-7) · [Nearing et al. 2022, *Nat Commun*](https://doi.org/10.1038/s41467-022-28034-z)

Here ANCOM-BC is run with scikit-bio's implementation. A pseudocount of 1 replaces zero counts, which the method requires, and effects are reported as log2 fold changes (+1 means twice as abundant) with 95% confidence intervals.

In [ ]:
add_differential_abundance(nasal, group="side", min_prevalence=0.25)
explore_forest(nasal, group="side")

In [20]:
add_differential_abundance(nasal, group="role", min_prevalence=0.25)
explore_forest(nasal, group="role")

Testing 96 of 605 genera for role
1 genus–comparison pairs with q < 0.05 (Holm-adjusted); full table in results/nasal/ancombc-role.tsv


interactive(children=(Dropdown(description='Comparison', options=('student vs instructor',), style=Description…

## Reflection

1. What did you predict for each question, and what did you find?
2. Which result would you defend in a paper, and which would you not? Why?
3. What do the blanks and the low-read samples tell you about the limits of these data?
4. If you could redesign the study, what would you change: number of people, swabs per nostril, controls, metadata?

**Read more:** [Knight et al. 2018, *Nat Rev Microbiol*](https://doi.org/10.1038/s41579-018-0029-9)